In [0]:
import dataiku
from dataikuapi.dss.dataset import DSSDatasetSettings

# Scope: Current Project
client = dataiku.api_client()
project = client.get_default_project()

# Enumerate all datasets
datasets = project.list_datasets()
print(f"Found {len(datasets)} datasets in project {project.project_key}")

for ds_item in datasets:
    ds_name = ds_item['name']
    print(f"Processing dataset: {ds_name}")
    
    dataset = project.get_dataset(ds_name)
    
    # 1. Clear the Schema
    try:
        # Get the settings handler
        settings = dataset.get_settings()
        
        # Check if schema exists before clearing to avoid unnecessary saves
        if settings.schema_columns:
            # Set schema columns to an empty list
            settings.schema_columns = []
            settings.save()
            print(f"  SUCCESS: Schema cleared.")
        else:
            settings.schema_columns = []
            settings.save()
            print(f"  SKIPPED: Schema was already empty.")
            
    except Exception as e:
        print(f"  ERROR: Failed to clear schema. Reason: {e}")

    # 2. Clear the Dataset (Data)
    try:
        # The clear() method throws an exception if the dataset type 
        # does not support clearing (e.g., some read-only external inputs)
        dataset.clear()
        print(f"  SUCCESS: Data cleared.")
    except Exception as e:
        # Capture specific API errors regarding capability
        print(f"  WARNING: Could not clear data (may not be supported for this type). Reason: {e}")

    print("-" * 30)